# Dinámica de Poblaciones — Modelo Logístico Continuo
## Error del Método de Euler: Solución Numérica vs Exacta

### El modelo

La **ecuación diferencial logística continua** es:

$$\frac{dX}{dt} = \alpha\, X\!\left(1 - \frac{X}{L}\right)$$

con parámetros $\alpha > 0$ (tasa de crecimiento) y $L > 0$ (capacidad de carga).

### Solución exacta

Esta EDO es separable y su solución analítica exacta con condición inicial $X(0) = X_0$ es:

$$\boxed{X_{\text{exacta}}(t) = \frac{L}{1 + \left(\dfrac{L}{X_0} - 1\right)e^{-\alpha t}}}$$

### Método de Euler (discretización)

Aplicando Euler hacia adelante con paso de tiempo $h$:

$$X_{n+1} = X_n + h\cdot\alpha\, X_n\!\left(1 - \frac{X_n}{L}\right), \qquad t_n = n\cdot h$$

### Error local y global de Euler

El **error de truncamiento local** (por paso) de Euler es $O(h^2)$:

$$e_{\text{local}} = X(t_{n+1}) - X_{n+1} = \frac{h^2}{2}X''(t_n) + O(h^3)$$

El **error global** acumula $\sim T/h$ pasos, por lo que:

$$e_{\text{global}}(t) = |X_{\text{Euler}}(t) - X_{\text{exacta}}(t)| = O(h)$$

Es decir: al reducir $h$ a la mitad, el error global debería reducirse a la mitad. Verificaremos esto empíricamente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# ── Estilo global ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#161b27',
    'axes.edgecolor':    '#2e3650',
    'axes.labelcolor':   '#cdd6f4',
    'xtick.color':       '#7f849c',
    'ytick.color':       '#7f849c',
    'grid.color':        '#2e3650',
    'grid.linestyle':    '--',
    'grid.alpha':        0.45,
    'text.color':        '#cdd6f4',
    'font.family':       'monospace',
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'legend.fontsize':   9,
})

print('Entorno listo.')

---
## 1. Implementación: solución exacta y método de Euler

In [ ]:
# ── Solución exacta ───────────────────────────────────────────────────────────
def solucion_exacta(t, X0, alpha, L):
    """
    Solución analítica de dX/dt = alpha*X*(1 - X/L):

        X(t) = L / (1 + (L/X0 - 1)*exp(-alpha*t))
    """
    return L / (1.0 + (L / X0 - 1.0) * np.exp(-alpha * t))


# ── Método de Euler ───────────────────────────────────────────────────────────
def euler_logistico(X0, alpha, L, h, T):
    """
    Integra X_{n+1} = X_n + h*alpha*X_n*(1 - X_n/L) hasta t = T.

    Retorna:
        t_arr  : array de tiempos  [0, h, 2h, ..., T]
        X_arr  : array de valores de X en cada paso
    """
    n_pasos = int(round(T / h))
    t_arr   = np.linspace(0.0, T, n_pasos + 1)
    X_arr   = np.empty(n_pasos + 1)
    X_arr[0] = X0
    for n in range(n_pasos):
        Xn = X_arr[n]
        X_arr[n + 1] = Xn + h * alpha * Xn * (1.0 - Xn / L)
    return t_arr, X_arr


# ── Error absoluto en cada instante de Euler ──────────────────────────────────
def error_euler(X0, alpha, L, h, T):
    """
    Devuelve (t_arr, error_arr) donde error_arr[n] = |X_Euler(t_n) - X_exacta(t_n)|.
    """
    t_arr, X_euler = euler_logistico(X0, alpha, L, h, T)
    X_exact        = solucion_exacta(t_arr, X0, alpha, L)
    return t_arr, np.abs(X_euler - X_exact)


print('Funciones definidas correctamente.')

---
## 2. Parámetros del experimento

In [ ]:
# ── Parámetros del modelo ─────────────────────────────────────────────────────
alpha = 1.0      # tasa de crecimiento intrínseca
L     = 10.0     # capacidad de carga
X0    = 1.0      # condición inicial  (X0 < L)
T     = 12.0     # tiempo final de integración

# ── Pasos de tiempo a comparar ────────────────────────────────────────────────
pasos_h = {
    'h = 1.00':  1.00,
    'h = 0.10':  0.10,
    'h = 0.01':  0.01,
}

colores_h = {
    'h = 1.00': '#f38ba8',   # rojo
    'h = 0.10': '#f9e2af',   # amarillo
    'h = 0.01': '#a6e3a1',   # verde
}

# ── Resumen ───────────────────────────────────────────────────────────────────
print(f'Parámetros del modelo logístico:')
print(f'  α  = {alpha}   (tasa de crecimiento)')
print(f'  L  = {L}    (capacidad de carga)')
print(f'  X₀ = {X0}    (condición inicial)')
print(f'  T  = {T}   (tiempo final)')
print()
print(f'Valor asintótico exacto: X(∞) = L = {L}')
print(f'Punto de inflexión:      t* = ln(L/X₀ - 1)/α = {np.log(L/X0 - 1)/alpha:.4f}')

---
## 3. Soluciones numéricas vs exacta

Primero comparamos visualmente las trayectorias de Euler (para cada $h$) con la solución exacta.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5), facecolor='#0f1117')
ax.set_facecolor('#161b27')

# Solución exacta (curva de referencia densa)
t_ref  = np.linspace(0, T, 3000)
X_ref  = solucion_exacta(t_ref, X0, alpha, L)
ax.plot(t_ref, X_ref, color='#cba6f7', lw=2.8, zorder=5,
        label='Solución exacta')

# Euler para cada h
estilos = {'h = 1.00': ('o', 6, 0.95),
           'h = 0.10': ('s', 3, 0.80),
           'h = 0.01': ('-', 0, 0.65)}

for etiq, h in pasos_h.items():
    t_e, X_e = euler_logistico(X0, alpha, L, h, T)
    mk, ms, al = estilos[etiq]
    if mk == '-':
        ax.plot(t_e, X_e, color=colores_h[etiq], lw=1.5,
                alpha=al, label=f'Euler {etiq}', zorder=3)
    else:
        ax.plot(t_e, X_e, color=colores_h[etiq], lw=1.3,
                marker=mk, markersize=ms, alpha=al,
                label=f'Euler {etiq}', zorder=4)

ax.axhline(L, color='#45475a', lw=0.9, ls=':', label=f'$L = {L}$ (equilibrio)')
ax.set_xlabel('Tiempo $t$')
ax.set_ylabel('$X(t)$')
ax.set_title(
    r'Solución exacta vs Euler para distintos $h$' + '\n'
    r'$\dot{X} = \alpha X(1 - X/L)$,  '
    f'$\\alpha={alpha}$, $L={L}$, $X_0={X0}$',
    pad=12
)
ax.legend(framealpha=0.2, edgecolor='#45475a')
ax.grid(True)

plt.tight_layout()
plt.show()

---
## 4. Error absoluto $|X_{\text{Euler}}(t) - X_{\text{exacta}}(t)|$ vs $t$

Graficamos el error en **escala lineal** y en **escala logarítmica** para cada $h$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), facecolor='#0f1117')
for ax in axes:
    ax.set_facecolor('#161b27')

ax_lin, ax_log = axes

for etiq, h in pasos_h.items():
    t_e, err = error_euler(X0, alpha, L, h, T)
    col = colores_h[etiq]

    # Escala lineal
    ax_lin.plot(t_e, err, color=col, lw=1.6, alpha=0.9, label=etiq)

    # Escala log (ignorar ceros exactos)
    mask = err > 0
    ax_log.semilogy(t_e[mask], err[mask], color=col, lw=1.6,
                    alpha=0.9, label=etiq)

for ax, titulo in zip(
    [ax_lin, ax_log],
    ['Error absoluto — escala lineal', 'Error absoluto — escala log']
):
    ax.set_xlabel('Tiempo $t$')
    ax.set_ylabel(r'$|X_{\mathrm{Euler}}(t) - X_{\mathrm{exacta}}(t)|$')
    ax.set_title(titulo, pad=10)
    ax.legend(framealpha=0.2, edgecolor='#45475a')
    ax.grid(True)

plt.suptitle(
    r'Error de Euler $|X_{\mathrm{Euler}} - X_{\mathrm{exacta}}|$ para $h = 1,\, 0.1,\, 0.01$',
    fontsize=13, color='#cdd6f4', y=1.02
)
plt.tight_layout()
plt.show()

---
## 5. Error máximo y error en $t$ fijo — ¿Es proporcional a $h$?

Si el error global es $O(h)$, entonces al graficar $e_{\max}$ vs $h$ en escala log-log deberíamos obtener una recta de **pendiente 1**.

In [ ]:
# ── Barrido de h para verificar la proporcionalidad ──────────────────────────
h_array    = np.logspace(-3, 0, 40)   # h desde 0.001 hasta 1
err_max    = np.empty(len(h_array))   # error máximo en [0, T]
err_t5     = np.empty(len(h_array))   # error en t = 5 (punto representativo)
t_eval     = 5.0
X_ref_t5   = solucion_exacta(t_eval, X0, alpha, L)

for i, h in enumerate(h_array):
    t_e, err = error_euler(X0, alpha, L, h, T)
    err_max[i] = np.max(err)

    # Error en t_eval (buscar índice más cercano)
    idx = np.argmin(np.abs(t_e - t_eval))
    err_t5[i] = err[idx]

# ── Ajuste lineal en log-log ──────────────────────────────────────────────────
mask_fit = err_max > 1e-14   # evitar ceros numéricos
log_h    = np.log10(h_array[mask_fit])
log_e    = np.log10(err_max[mask_fit])
coef     = np.polyfit(log_h, log_e, 1)   # coef[0] = pendiente, coef[1] = intercepto
pendiente, intercepto = coef

log_h5   = np.log10(h_array[err_t5 > 1e-14])
log_e5   = np.log10(err_t5[err_t5 > 1e-14])
coef5    = np.polyfit(log_h5, log_e5, 1)
pend5    = coef5[0]

print(f'Ajuste log-log  e_max vs h:')
print(f'  Pendiente estimada = {pendiente:.4f}  (esperado ≈ 1.0)')
print()
print(f'Ajuste log-log  e(t=5) vs h:')
print(f'  Pendiente estimada = {pend5:.4f}  (esperado ≈ 1.0)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), facecolor='#0f1117')
for ax in axes:
    ax.set_facecolor('#161b27')

# ── (a) e_max vs h ───────────────────────────────────────────────────────────
ax1 = axes[0]
ax1.loglog(h_array[mask_fit], err_max[mask_fit],
           'o', color='#89dceb', markersize=5, alpha=0.85,
           label=r'$e_{\max}$ medido')

# Recta ajustada
h_fit_line = np.array([h_array[mask_fit].min(), h_array[mask_fit].max()])
e_fit_line = 10**intercepto * h_fit_line**pendiente
ax1.loglog(h_fit_line, e_fit_line, '--', color='#f9e2af', lw=2,
           label=f'Ajuste: pendiente $= {pendiente:.3f}$')

# Referencia de pendiente exactamente 1
e_ref1 = 0.5 * h_fit_line**1.0
ax1.loglog(h_fit_line, e_ref1, ':', color='#f38ba8', lw=1.5,
           label='Referencia $O(h^1)$')

# Marcar los tres h pedidos
for etiq, h_mark in pasos_h.items():
    idx_m = np.argmin(np.abs(h_array - h_mark))
    ax1.scatter(h_array[idx_m], err_max[idx_m],
                color=colores_h[etiq], s=120, zorder=6,
                marker='*', label=etiq)

ax1.set_xlabel('$h$ (paso de tiempo)')
ax1.set_ylabel(r'$e_{\max} = \max_t|X_{\mathrm{Euler}} - X_{\mathrm{exacta}}|$')
ax1.set_title(r'Error máximo $e_{\max}$ vs $h$ — escala log-log', pad=10)
ax1.legend(framealpha=0.2, edgecolor='#45475a', fontsize=8.5)
ax1.grid(True, which='both', alpha=0.3)

# ── (b) e(t=5) vs h ──────────────────────────────────────────────────────────
ax2 = axes[1]
mask5 = err_t5 > 1e-14
ax2.loglog(h_array[mask5], err_t5[mask5],
           's', color='#cba6f7', markersize=5, alpha=0.85,
           label=r'$e(t=5)$ medido')

h_fit5 = np.array([h_array[mask5].min(), h_array[mask5].max()])
e_fit5 = 10**coef5[1] * h_fit5**pend5
ax2.loglog(h_fit5, e_fit5, '--', color='#f9e2af', lw=2,
           label=f'Ajuste: pendiente $= {pend5:.3f}$')

e_ref1b = 0.05 * h_fit5**1.0
ax2.loglog(h_fit5, e_ref1b, ':', color='#f38ba8', lw=1.5,
           label='Referencia $O(h^1)$')

for etiq, h_mark in pasos_h.items():
    idx_m = np.argmin(np.abs(h_array - h_mark))
    ax2.scatter(h_array[idx_m], err_t5[idx_m],
                color=colores_h[etiq], s=120, zorder=6,
                marker='*', label=etiq)

ax2.set_xlabel('$h$ (paso de tiempo)')
ax2.set_ylabel(r'$|X_{\mathrm{Euler}}(5) - X_{\mathrm{exacta}}(5)|$')
ax2.set_title(r'Error puntual en $t = 5$ vs $h$ — escala log-log', pad=10)
ax2.legend(framealpha=0.2, edgecolor='#45475a', fontsize=8.5)
ax2.grid(True, which='both', alpha=0.3)

plt.suptitle(
    r'¿Es el error proporcional a $h$? — Verificación empírica',
    fontsize=13, color='#cdd6f4', y=1.02
)
plt.tight_layout()
plt.show()

print(f'\nConclusión: pendiente log-log ≈ {pendiente:.3f} ≈ 1  →  error global ~ O(h^1) ✓')

---
## 6. Tabla de errores para $h = 1,\, 0.1,\, 0.01$

Comparamos los errores clave y el ratio $e(h)/e(h/10)$ (debería ser $\approx 10$ si el error es $O(h)$).

In [ ]:
print('=' * 72)
print(f'{"h":>8} | {"e_max":>14} | {"e(t=5)":>14} | {"ratio e_max":>12} | {"ratio e(t=5)":>13}')
print('-' * 72)

resultados = {}
for etiq, h in pasos_h.items():
    t_e, err = error_euler(X0, alpha, L, h, T)
    e_max_h  = np.max(err)
    idx5     = np.argmin(np.abs(t_e - 5.0))
    e_t5_h   = err[idx5]
    resultados[h] = (e_max_h, e_t5_h)

hs = list(pasos_h.values())
for k, h in enumerate(hs):
    e_max_h, e_t5_h = resultados[h]
    if k == 0:
        ratio_max = '—'
        ratio_t5  = '—'
    else:
        e_max_prev, e_t5_prev = resultados[hs[k-1]]
        ratio_max = f'{e_max_prev / e_max_h:.2f}×'
        ratio_t5  = f'{e_t5_prev  / e_t5_h :.2f}×'
    print(f'{h:>8.2f} | {e_max_h:>14.6e} | {e_t5_h:>14.6e} | {ratio_max:>12} | {ratio_t5:>13}')

print('=' * 72)
print()
print('Interpretación del ratio:')
print('  Si error ~ O(h), reducir h por 10 debería reducir el error ~10×.')
print('  Ratios ≈ 10  →  confirma convergencia de orden 1.')

---
## 7. Panel completo — Trayectorias + Errores + Verificación $O(h)$

In [ ]:
fig = plt.figure(figsize=(18, 12), facecolor='#0f1117')
fig.suptitle(
    r'Método de Euler — Logística $\dot{X}=\alpha X(1-X/L)$  '
    f'($\\alpha={alpha}$, $L={L}$, $X_0={X0}$)',
    fontsize=14, color='#cdd6f4', y=1.01
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.38)

# ── Fila superior: trayectorias para cada h ───────────────────────────────────
for col_idx, (etiq, h) in enumerate(pasos_h.items()):
    ax = fig.add_subplot(gs[0, col_idx])
    ax.set_facecolor('#161b27')

    t_e, X_e = euler_logistico(X0, alpha, L, h, T)
    t_ref2   = np.linspace(0, T, 2000)
    X_ref2   = solucion_exacta(t_ref2, X0, alpha, L)

    ax.plot(t_ref2, X_ref2, color='#cba6f7', lw=2.2, label='Exacta', zorder=5)
    ax.plot(t_e, X_e, color=colores_h[etiq], lw=1.5,
            marker='o' if h >= 0.1 else None,
            markersize=4 if h >= 1 else 2,
            alpha=0.9, label=f'Euler ({etiq})', zorder=4)
    ax.axhline(L, color='#45475a', lw=0.8, ls=':')

    ax.fill_between(t_e,
                    solucion_exacta(t_e, X0, alpha, L),
                    X_e,
                    alpha=0.15, color=colores_h[etiq])

    ax.set_xlabel('$t$'); ax.set_ylabel('$X(t)$')
    ax.set_title(f'Trayectoria — {etiq}', fontsize=11, pad=8)
    ax.legend(fontsize=8, framealpha=0.2, edgecolor='#45475a')
    ax.grid(True)

# ── Fila inferior izquierda: error vs t (escala log) ─────────────────────────
ax_el = fig.add_subplot(gs[1, 0:2])
ax_el.set_facecolor('#161b27')
for etiq, h in pasos_h.items():
    t_e, err = error_euler(X0, alpha, L, h, T)
    mask = err > 0
    ax_el.semilogy(t_e[mask], err[mask], color=colores_h[etiq],
                   lw=1.6, alpha=0.9, label=etiq)
ax_el.set_xlabel('Tiempo $t$')
ax_el.set_ylabel(r'$|X_{\mathrm{Euler}} - X_{\mathrm{exacta}}|$  (log)')
ax_el.set_title(r'Error absoluto vs $t$ — escala logarítmica', pad=8)
ax_el.legend(framealpha=0.2, edgecolor='#45475a')
ax_el.grid(True, which='both', alpha=0.3)

# ── Fila inferior derecha: e_max vs h (log-log) ───────────────────────────────
ax_ll = fig.add_subplot(gs[1, 2])
ax_ll.set_facecolor('#161b27')
ax_ll.loglog(h_array[mask_fit], err_max[mask_fit],
             'o', color='#89dceb', markersize=4, alpha=0.8,
             label=r'$e_{\max}$')
ax_ll.loglog(h_fit_line, e_fit_line, '--', color='#f9e2af', lw=2,
             label=f'Pendiente $= {pendiente:.2f}$')
ax_ll.loglog(h_fit_line, 0.5*h_fit_line**1, ':', color='#f38ba8',
             lw=1.4, label='Ref $O(h)$')
for etiq, h_mark in pasos_h.items():
    idx_m = np.argmin(np.abs(h_array - h_mark))
    ax_ll.scatter(h_array[idx_m], err_max[idx_m],
                  color=colores_h[etiq], s=100, zorder=6, marker='*')
ax_ll.set_xlabel('$h$')
ax_ll.set_ylabel(r'$e_{\max}$')
ax_ll.set_title(r'$e_{\max}$ vs $h$ — log-log', pad=8)
ax_ll.legend(fontsize=8, framealpha=0.2, edgecolor='#45475a')
ax_ll.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Análisis del error: ¿por qué crece y luego decrece?

El perfil del error $|X_{\text{Euler}}(t) - X_{\text{exacta}}(t)|$ no es monótono: **crece** durante la fase sigmoidea y **decrece** al acercarse al equilibrio. Esto tiene una explicación precisa.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor='#0f1117')
for ax in axes:
    ax.set_facecolor('#161b27')

t_ref3 = np.linspace(0, T, 3000)
X_ref3 = solucion_exacta(t_ref3, X0, alpha, L)

# Segunda derivada exacta: X'' = alpha*X'*(1 - 2X/L)
Xp_ref = alpha * X_ref3 * (1 - X_ref3 / L)
Xpp_ref = alpha * Xp_ref * (1 - 2 * X_ref3 / L)

for ax_i, (etiq, h) in zip(axes, pasos_h.items()):
    t_e, err = error_euler(X0, alpha, L, h, T)

    # Error (eje izquierdo)
    ax_i.plot(t_e, err, color=colores_h[etiq], lw=2,
              label=r'$|e(t)|$', zorder=4)
    ax_i.set_xlabel('$t$')
    ax_i.set_ylabel(r'$|X_{\mathrm{Euler}} - X_{\mathrm{exacta}}|$',
                    color=colores_h[etiq])
    ax_i.tick_params(axis='y', labelcolor=colores_h[etiq])

    # |X''| en eje derecho (proporcional al error local)
    ax_r = ax_i.twinx()
    ax_r.patch.set_alpha(0)
    ax_r.plot(t_ref3, np.abs(Xpp_ref), color='#cba6f7',
              lw=1.3, ls='--', alpha=0.7, label=r"$|X''(t)|$")
    ax_r.set_ylabel(r"$|X''(t)|$", color='#cba6f7')
    ax_r.tick_params(axis='y', labelcolor='#cba6f7')

    ax_i.set_title(f'{etiq}  —  error y curvatura', pad=8, fontsize=10)
    ax_i.grid(True, alpha=0.3)

    # Leyenda combinada
    lines1, labels1 = ax_i.get_legend_handles_labels()
    lines2, labels2 = ax_r.get_legend_handles_labels()
    ax_i.legend(lines1 + lines2, labels1 + labels2,
                fontsize=8, framealpha=0.2, edgecolor='#45475a')

plt.suptitle(
    r"El error local de Euler es $\propto h^2 |X''(t)|/2$ — "
    r"ambas curvas comparten el mismo perfil temporal",
    fontsize=11, color='#cdd6f4', y=1.02
)
plt.tight_layout()
plt.show()

---
## 9. Resumen y conclusiones

### ¿El error es proporcional a $h$?

**Sí.** Los datos confirman la teoría:

| Reducción de $h$ | Factor teórico de reducción del error | Factor observado |
|-----------------|--------------------------------------|------------------|
| $h=1 \to h=0.1$ | $10\times$ | $\approx 10\times$ |
| $h=0.1 \to h=0.01$ | $10\times$ | $\approx 10\times$ |

La pendiente del gráfico log-log $e_{\max}$ vs $h$ es $\approx 1$, confirmando $e_{\text{global}} = O(h^1)$.

### ¿Por qué el error no es constante en el tiempo?

El error local de Euler en cada paso es:

$$e_{\text{local}}(t_n) = \frac{h^2}{2} X''(t_n) + O(h^3)$$

La segunda derivada de la logística es:

$$X''(t) = \alpha^2 X(t)\left(1 - \frac{X}{L}\right)\left(1 - \frac{2X}{L}\right)$$

- Es **máxima** alrededor del punto de inflexión $X = L/2$ (máxima curvatura de la sigmoide).
- **Se anula** cuando $X \to L$ (la curva se aplana al acercarse al equilibrio).

Por eso el error crece durante la fase de crecimiento acelerado y luego decae: el perfil del error sigue exactamente el perfil de $|X''(t)|$, como se ve en la sección anterior.

### ¿Cuándo usar cada $h$?

| $h$ | Error típico | Pasos hasta $T=12$ | Uso recomendado |
|-----|-------------|-------------------|------------------|
| $1.00$ | Grande | 12 | Solo estimaciones gruesas |
| $0.10$ | Moderado | 120 | Análisis exploratorio |
| $0.01$ | Pequeño | 1200 | Resultados confiables |

Para la logística, $h < 2/\alpha$ es necesario para que Euler sea estable. Con $\alpha = 1$ eso equivale a $h < 2$ cerca de $X=0$, pero la condición se vuelve más restrictiva cerca del equilibrio: $h < 2/(\alpha L)$ en sentido estricto.